# APT editor

Author: Maxime Rizzo, NASA GSFC
Date: 4/15/26

In [ ]:
import xml.etree.ElementTree as ET
import copy

In [ ]:
apt_file_dir = ''
tree = ET.parse(f'{apt_file_dir}APT_1022_SolarStrayLight_TSR.apt')
root = tree.getroot()

In [ ]:
ns = {'apt': 'http://www.stsci.edu/Roman/APT'}
for sp in root.iterfind('apt:SurveyPlan', ns):
    print(sp)

<Element '{http://www.stsci.edu/Roman/APT}SurveyPlan' at 0x1102253f0>


In [ ]:
### Scratch - for practice only ###
ns = {'apt': 'http://www.stsci.edu/Roman/APT'}
for sp in root.iterfind('apt:SurveyPlan', ns):
        for links in sp.iterfind('apt:Links', ns):
            for lr in links.iterfind('apt:LinkReq', ns):
                # print(lr.attrib.items(),lr.attrib['Type'], lr.attrib['Min'], lr.attrib['Max'])
                #print(lr.attrib['Type'], lr.attrib['Min'], lr.attrib['Max'])
                for ss in lr.iterfind('apt:SurveyStep', ns):
                    role = ss.get('Role')
                    step_number = ss.text
                    #print(role, step_number)

test = copy.deepcopy(lr)
test
for ss in test.iterfind('apt:SurveyStep', ns):
    role = ss.get('Role')
    if role=='Reference':
        ss.text = 20
    elif role=='Oriented':
        ss.text = 21


for ss in test.iterfind('apt:SurveyStep', ns):
    role = ss.get('Role')
    step_number = ss.text
    print(role, step_number)
print(lr.attrib.items())


i=0
for sp in root.iterfind('apt:SurveyPlan', ns):
        for links in sp.iterfind('apt:Links', ns):
            for lr in links.iterfind('apt:LinkReq', ns):
                 i+=1
                 print(i)

sp = root.find('apt:SurveyPlan', ns)
print(sp)
links = sp.find('apt:Links', ns)
print(links)
linkreq_list = links.findall('apt:LinkReq', ns)
print(linkreq_list)
tg = root.find('apt:Targets', ns)
targets = tg.findall('apt:FixedTarget', ns)
print(len(targets))
# for tgt in targets:
#      tg.remove(tgt)

# dat = load_targets(filename='RST_instrument_model/Pitch-raster.csv')
# for i in range(len(dat)):
#      new_tg = copy.deepcopy(target_model)
#      new_tg.find('apt:Number', ns).text = i+1
#      new_tg.find('apt:TargetName', ns).text = dat.iloc[i]['Name']
#      new_tg.find('apt:Comments', ns).text = dat.iloc[i]['Comments']
#      new_tg.find('apt:Category', ns).text = dat.iloc[i]['Category']
#      new_tg.find('apt:Keywords', ns).text = dat.iloc[i]['Description']
#      new_tg.find('apt:EquatorialCoordinates', ns).text = f"{dat.iloc[i]['RA']} {dat.iloc[i]['DEC']}"
#      tg.append(new_tg)
     
pp = root.find('apt:PassPlans', ns)
pp_list = pp.findall('apt:PassPlan', ns)
pp_target_only = copy.deepcopy(pp_list[0])
pp_prism_first = copy.deepcopy(pp_list[1])
pp_prism_last = copy.deepcopy(pp_list[2])


def list_fixed_target_attributes(fixed_target):
    attributes = [
        fixed_target.find('apt:Number', ns).text,
        fixed_target.find('apt:TargetName', ns).text,
        fixed_target.find('apt:Comments', ns).text,
        fixed_target.find('apt:RAProperMotion', ns).text,
        fixed_target.find('apt:DecProperMotion', ns).text,
        fixed_target.find('apt:RAProperMotionUnits', ns).text,
        fixed_target.find('apt:DecProperMotionUnits', ns).text,
        fixed_target.find('apt:Epoch', ns).text,
        fixed_target.find('apt:AnnualParallax', ns).text,
        fixed_target.find('apt:Category', ns).text,
        fixed_target.find('apt:Keywords', ns).text,
        fixed_target.find('apt:EquatorialCoordinates', ns).get('Value')
    ]
    return attributes

# list_fixed_target_attributes(target_model)

In [ ]:
sp = root.find('apt:SurveyPlan', ns)
print(sp)
for step in sp.iterfind('apt:SurveyPlanStep', ns):
    print(step)

In [ ]:
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord

def load_targets(filename='Pitch-raster2.csv'):

    dat = pd.read_csv(filename,  header=None, index_col=False, names=['Name', 'Category', 'Description', 'RA', 'DEC', 'Comments', 'Pitch', 'Roll', 'Sep'])

    # Convert RA and DEC to SkyCoord objects
    coords = SkyCoord(ra=dat['RA'].values*u.degree, dec=dat['DEC'].values*u.degree)

    # Convert to hmsdms strings with 4 decimal points and sign for declination
    dat['RAstr'] = coords.ra.to_string(unit=u.hour, sep=' ', precision=4, pad=True)
    dat['DECstr'] = coords.dec.to_string(unit=u.degree, sep=' ', precision=2, pad=True, alwayssign=True)

    return dat



In [ ]:
# dat = pd.read_csv('Pitch-raster2.csv',  header=None, index_col=False, names=['Name', 'Category', 'Description', 'RA', 'DEC', 'Comments', 'Pitch', 'Roll'])
# dat

In [ ]:
dat.iloc[10]['DEC']

In [ ]:
filename = 'APT_1022_SolarStrayLight_TSR.apt'
apt_file_dir = ''
ns = {'apt': 'http://www.stsci.edu/Roman/APT'}

target_file = 'Pitch-raster5_v2.csv'

def update_stray_light_APT(filename, rolls=[-14.9, -7.4, 0.0, 7.4, 14.9], apt_file_dir=apt_file_dir, orient_range=0.0):
    tree = ET.parse(f'{apt_file_dir}{filename}')
    root = tree.getroot()    
    n_orient= len(rolls)

    #### build target list correctly ####
    # assumes first target exists and is correctly built
    tg = root.find('apt:Targets', ns)
    targets = tg.findall('apt:FixedTarget', ns)

    # grab first target as model
    # make sure this one has the boresight correctly set
    target_model = copy.deepcopy(targets[0])
    # delete all targets
    for tgt in targets:
        tg.remove(tgt)

    # Load target file
    dat = load_targets(filename=target_file)
    n_targets = len(dat)

    # add targets one by one
    for i in range(len(dat)):
        new_tg = copy.deepcopy(target_model)
        new_tg.find('apt:Number', ns).text = str(i+1)
        new_tg.find('apt:TargetName', ns).text = dat.iloc[i]['Name']
        new_tg.find('apt:Comments', ns).text = dat.iloc[i]['Comments']
        new_tg.find('apt:Category', ns).text = dat.iloc[i]['Category']
        new_tg.find('apt:Keywords', ns).text = dat.iloc[i]['Description']
        new_tg.find('apt:EquatorialCoordinates', ns).attrib['Value'] = f"{dat.iloc[i]['RAstr']} {dat.iloc[i]['DECstr']}"
        tg.append(new_tg)

    #### Now build pass plans correctly ####
    # Assumes first 3 pass plans exist and are correctly built: one with just the target, one with the prism first, and one with the prism last
    pp = root.find('apt:PassPlans', ns)
    pp_list = pp.findall('apt:PassPlan', ns)
    pp_target_only = copy.deepcopy(pp_list[0])
    pp_prism_first = copy.deepcopy(pp_list[1])
    pp_prism_last = copy.deepcopy(pp_list[2])
    for p in pp_list:
        pp.remove(p)
    
    # now create the pass plans back, pointing at correct targets and with correct names
    for i in range(n_targets):
        first = copy.deepcopy(pp_target_only)
        second = copy.deepcopy(pp_prism_first)
        third = copy.deepcopy(pp_prism_last)

        first.attrib['Number'] = str(i*3+1)
        second.attrib['Number'] = str(i*3+2)
        third.attrib['Number'] = str(i*3+3)

        first.find('apt:Label', ns).text = f'Target_{i+1}'
        second.find('apt:Label', ns).text = f'Target_{i+1}_wPrismFirst'
        third.find('apt:Label', ns).text = f'Target_{i+1}_wPrismLast'
        first.find('apt:TargetSelection', ns).text = f'Fixed: {i+1}'
        second.find('apt:TargetSelection', ns).text = f'Fixed: {i+1}'
        third.find('apt:TargetSelection', ns).text = f'Fixed: {i+1}'

        pp.append(first)
        pp.append(second)
        pp.append(third)

    #### Now create the survey ####
    sp = root.find('apt:SurveyPlan', ns)
    steps = sp.findall('apt:SurveyPlanStep', ns)
    step_model = copy.deepcopy(steps[0])
    for step in steps:
        sp.remove(step)
    
    # for each orient in roll, observe the target
    for i in range(n_targets):

        roll_list = copy.deepcopy(rolls)

        if i%2 == 1:
            roll_list = roll_list[::-1]

        first_step = copy.deepcopy(step_model)
        first_step.find('apt:PassPlan', ns).text = str(i*3+2)
        s = first_step.find('apt:SpecialRequirements', ns)
        if dat.iloc[i]['DEC']<0: # flip the signs for continuity of schedulability
            s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']+roll_list[0]-orient_range:.4f} Degrees"
            s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']+roll_list[0]+orient_range:.4f} Degrees"
        else:
            s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']-roll_list[0]-orient_range:.4f} Degrees"
            s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']-roll_list[0]+orient_range:.4f} Degrees"
        
        sp.append(first_step)

        for j in range(n_orient-2):
            new_step = copy.deepcopy(step_model)
            new_step.find('apt:PassPlan', ns).text = str(i*3+1)
            s = new_step.find('apt:SpecialRequirements', ns)
            if dat.iloc[i]['DEC']<0:
                s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']+roll_list[1+j]-orient_range:.4f} Degrees"
                s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']+roll_list[1+j]+orient_range:.4f} Degrees"
            else:
                s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']-roll_list[1+j]-orient_range:.4f} Degrees"
                s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']-roll_list[1+j]+orient_range:.4f} Degrees"
            sp.append(new_step)
        
        last_step = copy.deepcopy(step_model)
        last_step.find('apt:PassPlan', ns).text = str(i*3+3)
        s = last_step.find('apt:SpecialRequirements', ns)
        if dat.iloc[i]['DEC']<0:
            s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']+roll_list[-1]-orient_range:.4f} Degrees"
            s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']+roll_list[-1]+orient_range:.4f} Degrees"
        else:
            s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']-roll_list[-1]-orient_range:.4f} Degrees"
            s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']-roll_list[-1]+orient_range:.4f} Degrees"
        sp.append(last_step)

    num_steps = 0
    for sp in root.iterfind('apt:SurveyPlan', ns):
        for step in sp.iterfind('apt:SurveyPlanStep', ns):
            num_steps+=1
    print(f'Number of survey steps: {num_steps}')

    links = sp.find('apt:Links', ns)

    # copy the first linking requirement as our model
    # lr_list = links.findall('apt:LinkReq', ns)
    # lr_model = copy.deepcopy(lr_list[0])

    # clean up all Linking requirements to start fresh
    for item in links.findall('apt:LinkReq', ns):
        links.remove(item)

    # now create sets of linking requirements between each n_orient+1 targets
    ### OBSOLETE - an earlier iteration was doing this but we are switching to absolute v3pa instead
    # for nsteps in range(num_steps//(n_orient+1)):
    #     for orient in range(n_orient):
    #         added_req = copy.deepcopy(lr_model)
    #         added_req.attrib['Min'] = f"{orient_list[orient][0]} Degrees"
    #         added_req.attrib['Max'] = f"{orient_list[orient][1]} Degrees"
    #         for ss in added_req.iterfind('apt:SurveyStep', ns):
    #             role = ss.get('Role')
    #             if role=='Reference':
    #                 ss.text = str(nsteps*(n_orient+1)+orient+1)
    #             elif role=='Oriented':
    #                 ss.text = str(nsteps*(n_orient+1)+orient+2)
            
    #         links.append(added_req)


    return tree
    

tree = update_stray_light_APT(filename, rolls=[-14.5, -7.4, 0.0, 7.4, 14.5], apt_file_dir=apt_file_dir)
    
tree.write(f'{apt_file_dir}APT_118_SolarStrayLight_TSR_final_v2.apt')

In [ ]:
filename = 'APT_1023_GhostsStrayLight_CAR171.apt'
apt_file_dir = ''

def load_targets_171(filename='CAR171_targets.csv'):

    dat = pd.read_csv(filename)

    # Convert RA and DEC to SkyCoord objects
    coords = SkyCoord(ra=dat['RA'].values*u.degree, dec=dat['DEC'].values*u.degree)

    # Convert to hmsdms strings with 4 decimal points and sign for declination
    dat['RA'] = coords.ra.to_string(unit=u.hour, sep=' ', precision=4, pad=True)
    dat['DEC'] = coords.dec.to_string(unit=u.degree, sep=' ', precision=2, pad=True, alwayssign=True)

    return dat


def update_stray_light_APT_171(filename, v3pa=142.2123, apt_file_dir=apt_file_dir, orient_range=0.0):
    tree = ET.parse(f'{apt_file_dir}{filename}')
    root = tree.getroot()    

    #### build target list correctly ####
    # assumes first target exists and is correctly built
    tg = root.find('apt:Targets', ns)
    targets = tg.findall('apt:FixedTarget', ns)

    # grab first target as model
    # make sure this one has the boresight correctly set
    target_model = copy.deepcopy(targets[0])
    # delete all targets
    for tgt in targets:
        tg.remove(tgt)

    # Load target file
    dat = load_targets_171()
    n_targets = len(dat)

    # add targets one by one
    for i in range(len(dat)):
        new_tg = copy.deepcopy(target_model)
        new_tg.find('apt:Number', ns).text = str(i+1)
        new_tg.find('apt:TargetName', ns).text = dat.iloc[i]['Name']
        new_tg.find('apt:Category', ns).text = 'Calibration'
        new_tg.find('apt:Keywords', ns).text = 'Stray light test'
        new_tg.find('apt:EquatorialCoordinates', ns).attrib['Value'] = f"{dat.iloc[i]['RA']} {dat.iloc[i]['DEC']}"
        tg.append(new_tg)

    #### Now build pass plans correctly ####
    pp = root.find('apt:PassPlans', ns)
    pp_list = pp.findall('apt:PassPlan', ns)
    pp_target = copy.deepcopy(pp_list[0])

    for p in pp_list:
        pp.remove(p)
    
    # now create the pass plans back, pointing at correct targets and with correct names
    for i in range(n_targets):
        first = copy.deepcopy(pp_target)

        first.attrib['Number'] = str(i+1)

        first.find('apt:Label', ns).text = f'Target_{i+1}'
        first.find('apt:TargetSelection', ns).text = f'Fixed: {i+1}'

        pp.append(first)
 
    #### Now create the survey ####
    sp = root.find('apt:SurveyPlan', ns)
    steps = sp.findall('apt:SurveyPlanStep', ns)
    step_model = copy.deepcopy(steps[0])

    # first clean up
    for step in steps:
        sp.remove(step)
    
    # observe the target
    for i in range(n_targets):

        first_step = copy.deepcopy(step_model)
        first_step.find('apt:PassPlan', ns).text = str(i+1)
        s = first_step.find('apt:SpecialRequirements', ns)
        s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{v3pa-orient_range:.4f} Degrees"
        s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{v3pa+orient_range:.4f} Degrees"
        
        sp.append(first_step)

    num_steps = 0
    for sp in root.iterfind('apt:SurveyPlan', ns):
        for step in sp.iterfind('apt:SurveyPlanStep', ns):
            num_steps+=1
    print(f'Number of survey steps: {num_steps}')

    return tree

tree = update_stray_light_APT_171(filename, v3pa=142.2123)

tree.write(f'{apt_file_dir}APT_1023_GhostsStrayLight_CAR171_modified.apt')